# Implement LDA to build a Topic Model

Notes:

I am following along with my M08 LDA with SKLearn Notes and my M08 HW as I complete this section.

## Setup

### Import Libraries

In [91]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation as LDA

In [92]:
import plotly.express as px
import plotly.io as pio

sns.set_theme(style='white')
pio.renderers.default = 'vscode'

### Configuration

In [93]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [94]:
# set chapter as bag
bag = "CHAPS"

In [95]:
colors = "YlGnBu"

### Load LIB and CORPUS

In [96]:
# load in tables
LIB = pd.read_csv('data/LIB.csv', sep='\t').set_index('book_id')
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)

## Prepare Data

### Build DOCS

SKLearn expects an F1 style corpus. So I create one from the annotated CORPUS table and keep only regular nouns.

In [97]:
# only regular nouns
DOCS = CORPUS[CORPUS.pos.str.match(r'^NNS?$')]\
    .groupby(bags[bag]).term_str\
    .apply(lambda x: ' '.join(x))\
    .to_frame('doc_str')

# DOCS

### Build Count Matrix (DTM)

Now I use SKLearn's CountVectorizer to convert the F1 corpus of chapters into a document-term vector space of word counts.

In [98]:
# instantiate CountVectorizer
count_engine = CountVectorizer(
    max_features=4000,
    max_df=.6, # drop terms appearing in more than 60% of chapters
    min_df=10, # drop terms appearing in fewer than 10 chapters (remove rare terms that would add noise without contributing to structure; hapax legomena etc)
    stop_words='english'
)

# fit the engine to the documents
# keep in mind that CountVectorizer outputs a sparse matrix
count_model = count_engine.fit_transform(DOCS['doc_str'])

# extract vocabulary array for use in PHI table
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'

# convert sparse matrix into dense Pandas dataframe
DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS) # convert to DataFrame to get DTM

DTM

ability  absence  accent  accident  \
book_id                      chap_num                                       
giants-bread                 0               0        0       0         0   
                             1               0        0       0         0   
                             2               0        0       0         0   
                             3               0        1       0         0   
                             4               0        0       0         0   
...                                        ...      ...     ...       ...   
the-seven-dials-mystery      31              0        0       0         0   
                             32              0        0       0         0   
                             33              0        0       0         1   
                             34              0        0       0         0   
the-tragedy-at-marsdon-manor 1               0        0       0         1   

                                       accomplice  account  accounts  \
book_id                      chap_num                                  
giants-bread                 0                  0        0         0   
                             1                  0        0         0   
                             2                  0        0         0   
                             3                  0        0         0   
                             4                  0        0         0   
...                                           ...      ...       ...   
the-seven-dials-mystery      31                 0        0         0   
                             32                 0        0         0   
                             33                 0        0         0   
                             34                 0        0         0   
the-tragedy-at-marsdon-manor 1                  0        0         0   

                                       accuracy  acquaintance  acquaintances  \
book_id                      chap_num                                          
giants-bread                 0                0             0              0   
                             1                0             0              0   
                             2                0             0              0   
                             3                0             0              0   
                             4                0             0              0   
...                                         ...           ...            ...   
the-seven-dials-mystery      31               0             0              0   
                             32               0             0              0   
                             33               0             0              0   
                             34               0             0              0   
the-tragedy-at-marsdon-manor 1                0             0              0   

                                       ...  world  wound  wrist  writing  \
book_id                      chap_num  ...                                 
giants-bread                 0         ...      1      0      0        0   
                             1         ...      5      0      0        0   
                             2         ...      1      0      0        0   
                             3         ...      3      0      0        0   
                             4         ...      3      0      1        0   
...                                    ...    ...    ...    ...      ...   
the-seven-dials-mystery      31        ...      0      0      0        0   
                             32        ...      2      0      0        0   
                             33        ...      0      1      0        0   
                             34        ...      0      0      0        1   
the-tragedy-at-marsdon-manor 1         ...      0      1      0        0   

                                       yards  yawn  year  years  yeste

## Fit LDA Model

In [99]:
# set parameters
n_components = 15 # number of topics
random_state = 36
n_top_terms = 10 # changed from 5 for better interpretibility

In [100]:
# instantiate LDA model
lda_engine = LDA(
    n_components = n_components,
    random_state = random_state
)

## Extract Results

### THETA (Document-Topic Matrix)

THETA: given this document, what is the probability distribution over topics? Each row is a document, each column is a topic, each cell is p(topic | document).

### PHI (Topic-Word Matrix)

PHI: given this topic, what is the probability distribution over words? Each row is a topic, each column is a term, each cell is p(word | topic).

## Save THETA and PHI and DOCS to CSV

In [101]:
# save the DOCS table to csv
DOCS.to_csv('data/LDA_DOCS.csv', sep='\t', index=True)

# save the THETA table to csv
THETA_raw.to_csv('data/LDA_THETA.csv', sep='\t', index=True)

# save the PHI table to csv
PHI_raw.to_csv('data/LDA_PHI.csv', sep='\t', index=True)

NameError: name 'THETA_raw' is not defined

## Load and Analyze

### Load THETA and PHI from CSV

In [ ]:
# after above was run intially, for reproducibility and to be safe and deterministic, now just load in THETA and PHI
THETA = pd.read_csv('data/LDA_THETA.csv', sep='\t').set_index(['book_id', 'chap_num'])
THETA.columns = THETA.columns.astype(int)
PHI = pd.read_csv('data/LDA_PHI.csv', sep='\t').set_index('topic_id')
PHI.columns.name = 'term_str'

In [ ]:
# sanity checks
assert THETA.shape[1] == n_components, "THETA topic count mismatch"
assert abs(THETA.sum(axis=1).mean() - 1.0) < 0.01, "THETA rows don't sum to 1"
print(f"THETA: {THETA.shape} ok")
print(f"PHI: {PHI.shape} ok")

THETA: (325, 15) ok
PHI: (15, 1406) ok


### TOPICS (Top Terms per Topic)

Creating a TOPICS table with top 10 words per topic (the top 5 are the first 5).

In [ ]:
TOPICS = PHI.stack().groupby('topic_id').apply(
    lambda x: ' '.join(x.sort_values(ascending=False)
    .head(n_top_terms).reset_index()['term_str'])
).to_frame('top_terms')

TOPICS['doc_weight_sum'] = THETA.sum(axis=0)
TOPICS['doc_weight_mean'] = THETA.mean(axis=0)

### Parameter Exploration Notes

1. max_features=4000, max_df=0.9, min_df=10, n_components = 20
- have a lot of generic physical and spatial vocabulary (door, room, head, eyes, hand, face) in T0, T2, T7, T14, and T15

2. I am going to lower max_df from 0.9 to 0.6 to see if lowering the threshold will eliminate some of these more generic terms from the vocab.
- This has gotten better. I've noticed some french showing up in T5. I am going to drop to n_components = 15 now.

3. I want cleaner separation than I have. I am going to try n_components 20 again now that I am looking at 10 terms in each.

I also ran some experiments and determined I should add the custom stops: 
custom_stops = ['friend', 'girl', 'table', 'men', 'lady', 'minutes']

In the end I have removed the custom stops and returned to the 15 components.

### Inspect Topics

In [ ]:
for topic_id, row in TOPICS.sort_values('doc_weight_mean', ascending=False).iterrows():
    print(f"T{str(topic_id).zfill(2)} ({row['doc_weight_mean']:.3f}): {row['top_terms']}")

T01 (0.138): friend word girl car letter train manner fellow hands window
T03 (0.134): girl doctor table minutes friend lady bed window floor end
T11 (0.124): boy father lot years mother bit girl lady sort word
T13 (0.094): murder crime body magistrate police inspector window husband point murderer
T00 (0.092): inspector window sir doctor table friend evening minutes question chair
T04 (0.077): girl mother money boy sister years love child letter father
T06 (0.077): train music compartment story love years world diamonds money race
T12 (0.062): chimneys letters business memoirs gentleman work bundle party hotel manuscript
T02 (0.046): father millionaire secretary daughter train jewels girl lady sir car
T10 (0.037): cabin deck evidence prisoner friend coffee strychnine paper point murder
T09 (0.037): police car friend lady wife men sir inspector ami road
T14 (0.025): sir friend key paper coffee letter mistress window evidence crime
T08 (0.024): death doctor years mistress friend sir mon

In [ ]:
# inspect all 15 topics
TOPICS.sort_values('doc_weight_mean', ascending=False)

,top_terms,doc_weight_sum,doc_weight_mean
topic_id,,,
1,friend word girl car letter train manner fello...,44.754787,0.137707
3,girl doctor table minutes friend lady bed wind...,43.669137,0.134367
11,boy father lot years mother bit girl lady sort...,40.155596,0.123556
13,murder crime body magistrate police inspector ...,30.471220,0.093758
0,inspector window sir doctor table friend eveni...,29.815032,0.091739
4,girl mother money boy sister years love child ...,25.079122,0.077167
6,train music compartment story love years world...,24.947769,0.076762
12,chimneys letters business memoirs gentleman wo...,20.230193,0.062247
2,father millionaire secretary daughter train je...,14.883821,0.045796


In [ ]:
# look at the top 5 topics
TOPICS.sort_values('doc_weight_mean', ascending=False).head(5)

,top_terms,doc_weight_sum,doc_weight_mean
topic_id,,,
1,friend word girl car letter train manner fello...,44.754787,0.137707
3,girl doctor table minutes friend lady bed wind...,43.669137,0.134367
11,boy father lot years mother bit girl lady sort...,40.155596,0.123556
13,murder crime body magistrate police inspector ...,30.471220,0.093758
0,inspector window sir doctor table friend eveni...,29.815032,0.091739


In [ ]:
# add labels and themes to TOPICS for riffs
# I had Gemini and Claude give me ideas for these iteratively
topic_data = {
    0:  {'label': 'Consultation and Inquiry',          'theme': 'Investigation'},
    1:  {'label': 'General Social Narrative',        'theme': 'Prose Infrastructure'},
    2:  {'label': 'Wealthy Estate Settings',         'theme': 'Genre Trope'},
    3:  {'label': 'Domestic Scene Setting',         'theme': 'Prose Infrastructure'},
    4:  {'label': 'Inheritance and Family Finance',    'theme': 'Motive'},
    5:  {'label': 'Romantic / Sentimental Narrative', 'theme': 'Genre Trope'},
    6:  {'label': 'High-Value Theft and Travel',       'theme': 'Genre Trope'},
    7:  {'label': 'Temporal / Forensic Evidence',     'theme': 'Clues'},
    8:  {'label': 'Life History and Mortality',         'theme': 'Background'},
    9:  {'label': 'Poirot and Associates',          'theme': 'Investigation'},
    10: {'label': 'Method of Poisoning and Transit',   'theme': 'Clues'},
    11: {'label': 'Domestic and Family Units',         'theme': 'Prose Infrastructure'},
    12: {'label': 'Administrative Correspondence',    'theme': 'Narrative Structure'},
    13: {'label': 'Official Legal Investigation',    'theme': 'Investigation'},
    14: {'label': 'Private Papers and Evidence',       'theme': 'Clues'}
}

# map data to TOPICS
TOPICS['label'] = TOPICS.index.map(lambda x: topic_data[x]['label'])
TOPICS['theme'] = TOPICS.index.map(lambda x: topic_data[x]['theme'])

TOPICS[['label', 'theme']].sort_index()

,label,theme
topic_id,,
0,Consultation & Inquiry,Investigation
1,General Social Narrative,Prose Infrastructure
2,Wealthy Estate Settings,Genre Trope
3,Medical / Physical Discovery,Narrative Structure
4,Inheritance & Family Finance,Motive
5,Romantic / Sentimental Narrative,Genre Trope
6,High-Value Theft & Travel,Genre Trope
7,Temporal / Forensic Evidence,Clues
8,Life History & Mortality,Background


### Explore Topic Distributions by Feature (Book/Sleuth/Work Type)

In [ ]:
THETAX = THETA.join(LIB[['sleuth', 'work_type', 'genre']], on='book_id').reset_index().set_index(['book_id', 'chap_num'])
THETAX['primary_genre'] = THETAX['genre'].str.split('|').str[0]

# THETAX

In [ ]:
# groupby book_id ("how much does this book talk about this topic?")
TOPIC_BOOK = THETAX.groupby('book_id').mean(numeric_only=True)
TOPIC_BOOK.T.style.background_gradient(axis=0, cmap="YlGnBu")

book_id,giants-bread,the-adventure-of-the-cheap-flat,the-adventure-of-the-egyptian-tomb,the-adventure-of-the-italian-nobleman,the-adventure-of-the-western-star,the-big-four,the-case-of-the-missing-will,the-disappearance-of-mr-davenheim,the-jewel-robbery-at-the-grand-metropolitan,the-kidnapped-prime-minister,the-man-in-the-brown-suit,the-million-dollar-bond-robbery,the-murder-at-the-vicarage,the-murder-of-roger-ackroyd,the-murder-on-the-links,the-mysterious-affair-at-styles,the-mystery-of-hunters-lodge,the-mystery-of-the-blue-train,the-secret-adversary,the-secret-of-chimneys,the-seven-dials-mystery,the-tragedy-at-marsdon-manor
0,0.014752,0.534072,0.000155,0.557670,0.071448,0.143512,0.000189,0.000133,0.000138,0.000115,0.027693,0.080345,0.062480,0.541626,0.004116,0.000197,0.268169,0.037691,0.051124,0.071730,0.067830,0.156513
1,0.024826,0.000163,0.000155,0.000163,0.203828,0.228927,0.112255,0.259307,0.000138,0.000115,0.191099,0.000245,0.082644,0.049752,0.246546,0.073769,0.000151,0.111225,0.156586,0.198606,0.168082,0.000164
2,0.000313,0.000163,0.000155,0.000163,0.000105,0.001308,0.000189,0.000133,0.000138,0.000115,0.011090,0.000245,0.000445,0.000353,0.057920,0.000197,0.000151,0.276188,0.068554,0.000387,0.024948,0.000164
3,0.062218,0.347907,0.000155,0.292349,0.392955,0.304446,0.000189,0.176884,0.272718,0.000115,0.119167,0.248863,0.047460,0.026027,0.051919,0.079201,0.000151,0.032212,0.518440,0.099334,0.167271,0.577568
4,0.225221,0.000163,0.000155,0.000163,0.000105,0.004333,0.651955,0.148978,0.000138,0.000115,0.030871,0.000245,0.061742,0.255586,0.087431,0.012595,0.123844,0.074571,0.025979,0.028250,0.017630,0.000164
5,0.038854,0.000163,0.000155,0.000163,0.000105,0.000279,0.000189,0.000133,0.000138,0.000115,0.061396,0.000245,0.000445,0.002306,0.000357,0.000197,0.000151,0.000510,0.001883,0.000387,0.029655,0.000164
6,0.203984,0.112241,0.000155,0.000163,0.039421,0.009183,0.000189,0.000133,0.000138,0.000115,0.246397,0.053777,0.004053,0.005689,0.034212,0.008446,0.000151,0.177038,0.012684,0.040835,0.005455,0.000164
7,0.006125,0.000163,0.000155,0.000163,0.000105,0.000279,0.000189,0.000133,0.000138,0.000115,0.006568,0.000245,0.000445,0.000597,0.000357,0.167525,0.000151,0.000510,0.000374,0.014808,0.096300,0.000164
8,0.000313,0.000163,0.997834,0.000163,0.000105,0.014164,0.000189,0.000133,0.000138,0.000115,0.007773,0.094242,0.009218,0.019675,0.023067,0.007495,0.000151,0.067390,0.001515,0.046512,0.015912,0.103222
9,0.006475,0.003983,0.000155,0.000163,0.128357,0.078594,0.031202,0.167562,0.518663,0.998391,0.019587,0.000245,0.039006,0.004827,0.007539,0.000197,0.375220,0.131967,0.001717,0.027569,0.002981,0.000164


In [ ]:
TOPIC_BOOK[[1, 3, 11, 13, 0]].style.background_gradient(cmap='YlGnBu', axis=0)

,1,3,11,13,0
book_id,,,,,
giants-bread,0.024826,0.062218,0.400549,0.000313,0.014752
the-adventure-of-the-cheap-flat,0.000163,0.347907,0.000163,0.000163,0.534072
the-adventure-of-the-egyptian-tomb,0.000155,0.000155,0.000155,0.000155,0.000155
the-adventure-of-the-italian-nobleman,0.000163,0.292349,0.000163,0.000163,0.557670
the-adventure-of-the-western-star,0.203828,0.392955,0.054773,0.000105,0.071448
the-big-four,0.228927,0.304446,0.007219,0.045017,0.143512
the-case-of-the-missing-will,0.112255,0.000189,0.000189,0.000189,0.000189
the-disappearance-of-mr-davenheim,0.259307,0.176884,0.000133,0.161376,0.000133
the-jewel-robbery-at-the-grand-metropolitan,0.000138,0.272718,0.000138,0.000138,0.000138


In [ ]:
# groupby sleuth ("how much do works with this sleuth talk about this topic?")
THETAX['sleuth'] = THETAX['sleuth'].fillna('None') # deal with giants bread
TOPIC_SLEUTH = THETAX.groupby('sleuth').mean(numeric_only=True)
TOPIC_SLEUTH.T.style.background_gradient(axis=0, cmap="YlGnBu")

sleuth,Colonel Race,Hercule Poirot,Miss Jane Marple,None,Superintendent Battle,Tommy and Tuppence
0,0.027693,0.153013,0.062480,0.014752,0.069690,0.051124
1,0.191099,0.134639,0.082644,0.024826,0.182640,0.156586
2,0.011090,0.087232,0.000445,0.000313,0.013234,0.068554
3,0.119167,0.091245,0.047460,0.062218,0.134870,0.518440
4,0.030871,0.099257,0.061742,0.225221,0.022695,0.025979
5,0.061396,0.000752,0.000445,0.038854,0.015697,0.001883
6,0.246397,0.059900,0.004053,0.203984,0.022329,0.012684
7,0.006568,0.016760,0.000445,0.006125,0.057434,0.000374
8,0.007773,0.038737,0.009218,0.000313,0.030505,0.001515
9,0.019587,0.065666,0.039006,0.006475,0.014708,0.001717


In [ ]:
TOPIC_SLEUTH[[1, 3, 11, 13, 0]].style.background_gradient(cmap='YlGnBu', axis=0)

,1,3,11,13,0
sleuth,,,,,
Colonel Race,0.191099,0.119167,0.054970,0.013461,0.027693
Hercule Poirot,0.134639,0.091245,0.028377,0.108084,0.153013
Miss Jane Marple,0.082644,0.047460,0.329122,0.312307,0.062480
None,0.024826,0.062218,0.400549,0.000313,0.014752
Superintendent Battle,0.182640,0.134870,0.166840,0.084553,0.069690
Tommy and Tuppence,0.156586,0.518440,0.046730,0.003416,0.051124


In [ ]:
# groupby work type ("how much do works of this type talk about this topic?")
TOPIC_WORK = THETAX.groupby('work_type').mean(numeric_only=True)
TOPIC_WORK.T.style.background_gradient(axis=0, cmap="YlGnBu")

work_type,novel,short_story
0,0.089637,0.151723
1,0.140695,0.052426
2,0.047395,0.000156
3,0.131717,0.209987
4,0.076921,0.084184
5,0.014604,0.000156
6,0.078793,0.018786
7,0.020402,0.000156
8,0.020936,0.108769
9,0.030815,0.202191


In [ ]:
# groupby primary genre ("how much do works of this primary genre talk about this topic?")
TOPIC_GENRE = THETAX.groupby('primary_genre').mean(numeric_only=True)
TOPIC_GENRE.T.style.background_gradient(axis=0, cmap="YlGnBu")

primary_genre,Detective,Espionage,Murder Mystery,Romance
0,0.143512,0.063962,0.112998,0.014752
1,0.228927,0.174602,0.127726,0.024826
2,0.001308,0.030301,0.065233,0.000313
3,0.304446,0.253206,0.068388,0.062218
4,0.004333,0.023708,0.088267,0.225221
5,0.000279,0.011435,0.012939,0.038854
6,0.009183,0.019353,0.092651,0.203984
7,0.000279,0.039831,0.013485,0.006125
8,0.014164,0.021562,0.029781,0.000313
9,0.078594,0.010700,0.050499,0.006475


In [ ]:
TOPIC_GENRE[[1, 3, 11, 13, 0]].style.background_gradient(cmap='YlGnBu', axis=0)

,1,3,11,13,0
primary_genre,,,,,
Detective,0.228927,0.304446,0.007219,0.045017,0.143512
Espionage,0.174602,0.253206,0.129785,0.059522,0.063962
Murder Mystery,0.127726,0.068388,0.088098,0.130743,0.112998
Romance,0.024826,0.062218,0.400549,0.000313,0.014752


## LDA + PCA Visualization (4)

Apply PCA to the THETA table and plot the topics in the space opened by the first two components.

Size the points based on the mean document weight of each topic (using the THETA table).

Color the points basd on a metadata feature from the LIB table.

Provide a brief interpretation of what you see.

(INSERT IMAGE HERE)

(INSERT INTERPRETATION HERE)

In [ ]:
# instantiate PCA engine with defaults except random_state=36 and n_components=2
pca_engine = PCA(n_components = 2, random_state = 36)

In [ ]:
# fit PCA on THETA
pca_engine.fit(THETA)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",2
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 

In [ ]:
# build a data frame with topic positions in PC space
# pca.components_.T has shape (n_topics, 2)
topic_coords = pd.DataFrame(
    pca_engine.components_.T,
    index=THETA.columns,       # topic_ids 0-14
    columns=['PC0', 'PC1']
)

# add size by mean document weight per topic
topic_coords['mean_weight'] = THETA.mean(axis=0)

# add color by sleuth
TOPIC_SLEUTH = THETAX.groupby('sleuth').mean(numeric_only=True)
topic_coords['dominant_sleuth'] = TOPIC_SLEUTH.idxmax(axis=0)

# or by genre
TOPIC_GENRE = THETAX.groupby('primary_genre').mean(numeric_only=True)
topic_coords['dominant_genre'] = TOPIC_GENRE.idxmax(axis=0)

# add top terms for hover
topic_coords['top_terms'] = TOPICS['top_terms']

In [ ]:
# plot
px.scatter(
    topic_coords,
    x='PC0', y='PC1',
    size='mean_weight',
    color='dominant_sleuth',
    hover_name=topic_coords.index,
    hover_data=['top_terms'],
    title='LDA Topics in PCA Space'
)

In [ ]:
# plot
px.scatter(
    topic_coords,
    x='PC0', y='PC1',
    size='mean_weight',
    color='dominant_genre',
    hover_name=topic_coords.index,
    hover_data=['top_terms'],
    title='LDA Topics in PCA Space'
)

In [ ]:
topic_coords.sort_values('PC1', ascending=False)[['PC1', 'top_terms', 'dominant_genre', 'dominant_sleuth']]

,PC1,top_terms,dominant_genre,dominant_sleuth
3,0.769509,girl doctor table minutes friend lady bed wind...,Detective,Tommy and Tuppence
0,0.263642,inspector window sir doctor table friend eveni...,Detective,Hercule Poirot
12,0.017592,chimneys letters business memoirs gentleman wo...,Espionage,Superintendent Battle
5,0.004578,music darling love doctor suit point diamonds ...,Romance,Colonel Race
14,0.004499,sir friend key paper coffee letter mistress wi...,Murder Mystery,Hercule Poirot
9,0.004259,police car friend lady wife men sir inspector ...,Detective,Hercule Poirot
7,0.000800,clocks oclock strychnine bed evidence alarm so...,Espionage,Superintendent Battle
8,-0.001235,death doctor years mistress friend sir money h...,Murder Mystery,Hercule Poirot
10,-0.019879,cabin deck evidence prisoner friend coffee str...,Detective,Colonel Race
2,-0.033049,father millionaire secretary daughter train je...,Murder Mystery,Hercule Poirot


### Interpretation

PC0 separates topic 9 from everything else. Topic 9 is heavily associated with Tommy and Tuppence.

PC1 separates within the remaining topics and appears to separate investigative vocabulary from action/thriller vocabulary. (Detection vs. Thriller)

REWRITE????

## Save TOPICS

In [ ]:
# save the TOPICS table to csv
TOPICS.to_csv('data/LDA_TOPICS.csv', sep='\t', index=True)